## 📦 Step 1: Install Dependencies

In [ ]:
# Install PyTorch Geometric and extensions
!pip -q install torch-geometric
!pip -q install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-2.6.0+cu121.html

# Install utilities
!pip -q install numpy pandas scipy scikit-learn tqdm wandb geopandas

print(" All dependencies installed!")

## 💾 Step 2: Mount Google Drive & Setup Paths

In [ ]:
from google.colab import drive
import os
import sys

# Mount Drive
drive.mount('/content/drive')

# ⚠️ IMPORTANT: Update this path to match YOUR Drive structure
BASE_PATH = "/content/drive/MyDrive/Zamin_thesis/ml_surrogates_for_agent_based_transport_models"

# Verify path exists
if not os.path.exists(BASE_PATH):
    print(f"❌ ERROR: Path not found: {BASE_PATH}")
    print("Please update BASE_PATH in the cell above!")
else:
    print(f"✅ Repository found: {BASE_PATH}")
    
    # Add to Python path
    sys.path.insert(0, os.path.join(BASE_PATH, "scripts"))
    print(f"✅ Scripts added to Python path")

## 🔍 Step 3: Verify Data Files

In [ ]:
import glob

# Check dataset directory
dataset_dir = os.path.join(BASE_PATH, "data/train_data/dist_not_connected_10k_1pct")

if not os.path.exists(dataset_dir):
    print(f"❌ Dataset directory not found: {dataset_dir}")
    print("\nPlease check your data folder structure!")
else:
    # Find all .pt files
    pt_files = sorted(glob.glob(os.path.join(dataset_dir, "datalist_batch_*.pt")))
    
    print(f"✅ Dataset directory found: {dataset_dir}")
    print(f"✅ Found {len(pt_files)} batch files")
    
    if pt_files:
        print(f"\nFirst few files:")
        for f in pt_files[:5]:
            print(f"  - {os.path.basename(f)}")
        if len(pt_files) > 5:
            print(f"  ... and {len(pt_files) - 5} more")
    else:
        print("⚠️ No batch files found! Please upload your .pt files.")

## 🎯 Step 4: Configure Training Parameters

**These are Elena's EXACT parameters from the paper (Section 6.3)**

In [ ]:
# Training configuration (Elena's paper settings)
config = {
    # Paths
    "dataset_path": os.path.join(BASE_PATH, "data/train_data/dist_not_connected_10k_1pct"),
    
    # Model architecture
    "gnn_arch": "point_net_transf_gat",  # Elena's model
    "in_channels": 5,  # 5 features (VOL, CAP, CAP_RED, SPEED, LENGTH)
    "use_all_features": False,  # Skip HIGHWAY feature
    
    # Training hyperparameters
    "num_epochs": 750,
    "lr": 0.0005,  # 5e-4
    "early_stopping_patience": 40,
    "batch_size": 8,
    "gradient_accumulation_steps": 3,  # Effective batch = 24
    
    # Regularization
    "use_dropout": True,
    "dropout": 0.3,
    "use_gradient_clipping": True,
    
    # Project settings
    "project_name": "TR-C_Benchmarks",
    "unique_model_description": "elena_colab_1000_scenarios",
    
    # Reproducibility
    "seed": 42
}

print("📊 Training Configuration:")
print("="*60)
for key, value in config.items():
    print(f"  {key:30s}: {value}")
print("="*60)

## 🎮 Step 5: WandB Setup (Optional but Recommended)

In [ ]:
import wandb

# Login to WandB
wandb.login()

print("✅ WandB authentication complete!")
print("You can monitor training at: https://wandb.ai")

## 🚀 Step 6: Run Training Using run_models.py

**Method 1: Direct Python Command** (Recommended)

In [ ]:
# Change to repository directory
os.chdir(BASE_PATH)
print(f"Working directory: {os.getcwd()}")

# Build command
cmd = f"""python scripts/training/run_models.py \
    --dataset_path "{config['dataset_path']}" \
    --gnn_arch {config['gnn_arch']} \
    --project_name {config['project_name']} \
    --unique_model_description {config['unique_model_description']} \
    --in_channels {config['in_channels']} \
    --use_all_features {config['use_all_features']} \
    --num_epochs {config['num_epochs']} \
    --batch_size {config['batch_size']} \
    --lr {config['lr']} \
    --early_stopping_patience {config['early_stopping_patience']} \
    --use_dropout {config['use_dropout']} \
    --dropout {config['dropout']} \
    --gradient_accumulation_steps {config['gradient_accumulation_steps']} \
    --use_gradient_clipping {config['use_gradient_clipping']} \
    --seed {config['seed']}
"""

print("🚀 Starting training...")
print("="*60)
print("This will take approximately 2-3 hours for 1000 scenarios")
print("Expected R² score: 0.72-0.76")
print("="*60)
print()

# Run the command
!{cmd}

## 📊 Step 7: Load and Evaluate Trained Model

In [ ]:
import torch
from gnn.models.point_net_transf_gat import PointNetTransfGAT
from gnn.help_functions import GNN_Loss, validate_model_during_training
import joblib

# Model path
model_path = os.path.join(
    BASE_PATH, 
    f"data/{config['project_name']}/{config['unique_model_description']}/trained_model/model.pth"
)

print(f"Loading model from: {model_path}")

if not os.path.exists(model_path):
    print("❌ Model file not found! Training may not have completed.")
else:
    # Create model instance
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    model = PointNetTransfGAT(
        in_channels=config['in_channels'],
        out_channels=1,
        dropout=config['dropout'],
        use_dropout=config['use_dropout'],
        predict_mode_stats=False,
        dtype=torch.float32,
        log_to_wandb=False
    ).to(device)
    
    # Load trained weights
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    print("✅ Model loaded successfully!")
    print(f"   Device: {device}")
    print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 📈 Step 8: Test Set Evaluation

In [ ]:
# Load test data and scalers
pipeline_path = os.path.join(
    BASE_PATH,
    f"data/{config['project_name']}/{config['unique_model_description']}"
)

test_loader = torch.load(
    os.path.join(pipeline_path, 'test_dl.pt'),
    map_location='cpu',
    weights_only=False
)

scalers_test = {
    'x_scaler': joblib.load(os.path.join(pipeline_path, 'test_x_scaler.pkl')),
    'pos_scaler': joblib.load(os.path.join(pipeline_path, 'test_pos_scaler.pkl'))
}

print(f"✅ Test data loaded: {len(test_loader)} batches")

# Create config for validation
class TestConfig:
    predict_mode_stats = False

test_config = TestConfig()

# Create loss function (dummy num_nodes for test)
loss_fn = GNN_Loss("mse", 31635, device, False)

# Evaluate
print("\n🔬 Running test evaluation...")
test_loss, r2, spearman, pearson = validate_model_during_training(
    config=test_config,
    model=model,
    dataset=test_loader,
    loss_func=loss_fn,
    device=device,
    scalers_validation=scalers_test
)

# Display results
print("\n" + "="*60)
print("📊 FINAL TEST RESULTS")
print("="*60)
print(f"  Test Loss:            {test_loss:.6f}")
print(f"  R² Score:             {r2:.4f}")
print(f"  Spearman Correlation: {spearman:.4f}")
print(f"  Pearson Correlation:  {pearson:.4f}")
print("="*60)

print("\n🎯 Comparison with Elena's Paper:")
print(f"  Elena (10k scenarios): R² = 0.76")
print(f"  Your model (1k):       R² = {r2:.4f}")

if r2 >= 0.72:
    print("\n  ✅ EXCELLENT! Model performs at paper level!")
elif r2 >= 0.65:
    print("\n  ✓ GOOD! Close to paper results!")
else:
    print("\n  ⚠️ Below expected. Consider training longer or with more data.")

## 📁 Step 9: Check Saved Files

In [ ]:
import glob

output_dir = os.path.join(
    BASE_PATH,
    f"data/{config['project_name']}/{config['unique_model_description']}"
)

print(f"📁 Output Directory: {output_dir}")
print("\nSaved Files:")
print("="*60)

# List all files recursively
for root, dirs, files in os.walk(output_dir):
    level = root.replace(output_dir, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        size = os.path.getsize(file_path) / (1024 * 1024)  # MB
        print(f"{subindent}{file} ({size:.2f} MB)")

## 🎉 Training Complete!

### 📊 What You Have Now:

1. **Trained Model**: Best model saved in `trained_model/model.pth`
2. **Checkpoints**: Every 20 epochs in `checkpoints/` folder
3. **Data Loaders**: Train, validation, test loaders saved
4. **Scalers**: Feature normalization scalers saved
5. **WandB Logs**: Training curves available at wandb.ai

### 🎯 Next Steps:

- **Use the model for predictions** on new scenarios
- **Analyze results by road type** (motorway, primary, etc.)
- **Compare different policies** using the trained model
- **Train with more data** (all 10k scenarios) for better R²

---

**Congratulations! You've successfully replicated Elena Boreale's GNN model! 🚀**